# Mây AI Studio trên Google Colab
Trò chuyện · Tạo ảnh · Dựng video bằng mô hình mã nguồn mở, chạy trên GPU miễn phí của Colab.

**Trước tiên:** menu *Runtime → Change runtime type → T4 GPU → Save*. Sau đó chạy lần lượt từng ô (bấm ▶).

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Lấy mã nguồn
Nếu repo của bạn để **công khai**, chạy ô dưới. Nếu repo **riêng tư**: tải thư mục `ai-studio` về máy dạng .zip rồi kéo vào khung Files bên trái, chạy `!unzip ai-studio.zip`.

In [ ]:
REPO = 'https://github.com/thanhdangkhoa16122008-hash/TDK'  # đổi nếu cần
BRANCH = 'ccr-b6e744b3-y0noaa'
!git clone -q --depth 1 -b $BRANCH $REPO tdk || echo 'Không clone được — xem hướng dẫn repo riêng tư ở trên'
%cd tdk/ai-studio

In [ ]:
# 3. Cài thư viện (~2–3 phút)
!pip install -q -r requirements.txt

## 4. Lưu mô hình & LoRA vào Google Drive (khuyên dùng)
Colab xóa mọi thứ khi tắt. Gắn Drive để lần sau không phải tải lại mô hình (~30 GB) và giữ LoRA đã huấn luyện.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/ai-studio/loras', exist_ok=True)
os.environ['HF_HOME'] = '/content/drive/MyDrive/ai-studio/hf-cache'
!rm -rf loras && ln -s /content/drive/MyDrive/ai-studio/loras loras

## 5. Chạy AI Studio
Đợi dòng `Running on public URL: https://….gradio.live` rồi bấm vào link. Lần đầu mỗi tính năng sẽ tải mô hình (chat ~6 GB, ảnh ~7 GB, video ~10–20 GB) nên chờ vài phút.

In [ ]:
!python app.py --share

---
# Huấn luyện thêm (tùy chọn)
Dừng ô 5 (■) trước khi huấn luyện để giải phóng GPU.

### A. Dạy Mây cách nói chuyện của bạn (LoRA chat)
Sửa/thêm hội thoại vào `data/chat_vi.jsonl` (mỗi dòng một hội thoại). Càng nhiều hội thoại chất lượng (300–1000) càng tốt. T4: ~10–40 phút.

In [ ]:
!python train/train_chat_lora.py --epochs 3

### B. Dạy phong cách ảnh / nhân vật riêng (LoRA ảnh)
Tải 15–40 ảnh vào `data/images/muro-var/` (khung Files bên trái), rồi chạy. T4: ~30–60 phút cho 800 bước.

In [ ]:
!python train/train_image_lora.py --images data/images/muro-var --name muro-var --prompt "murovar style, black shadow figure" --steps 800

Sau khi huấn luyện, chạy lại ô 5. LoRA chat tự được dùng; LoRA ảnh chọn trong tab **Tạo ảnh → Nâng cao**, và nhớ viết từ khóa (vd `murovar style`) trong mô tả.